# Movement-strategy validation

**Question.** On the 3-D mechanism model, how faithfully does each of the four
movement strategies reproduce a commanded path?

**Scope.** Standalone; the control strategy only. Everything is in *controller
units* (command counts). Actuator calibration - servo ticks, spool radius, dead
zone, latency - is out of scope and belongs to `validetion/Servomotor` and
`validetion/Servo+thimble`.

**Design.** All four strategies - `CARDINAL` (4-way), `CARDINAL_DIAGONAL`
(8-way), `FREE_FORM` (continuous) and `IK` (continuous) - are run on the same
3-D mechanism model, along the same commanded path (line out, one full circle,
line back).

**Limitation.** Commands are decoded with the model that generated them, so
this measures strategy-induced path error, not physical mechanism accuracy.
That the wire FK genuinely inverts the controller's IK path is verified in
`tests/test_wire_forward_kinematics.py`.

Computation lives in `analysis.py` and `figures.py`; this notebook calls them,
so there is exactly one implementation.

In [1]:
import pandas as pd

from analysis import StudyConfig, comparison_table, run_study
from figures import plot_motor_commands, plot_reconstructed_paths, plot_strategy_comparison

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)

config = StudyConfig()
samples, metrics = run_study(config)

print(f"model: {config.kinematic_model.value} | radius {config.radius:.0f} controller units | "
      f"{config.line_steps + config.circle_steps + config.return_steps + 1} samples per strategy")

model: ik | radius 160 controller units | 301 samples per strategy


## Strategy comparison

`total_rms` is the distance between where the tactor was commanded and where it
ended up, averaged over the path. It splits into `quantisation_rms` (what the
strategy's direction resolution costs) and `execution_rms` (what the model plus
integer command truncation cost).

In [2]:
comparison_table(metrics, config).round(3)

,quantisation,total_rms,total_max,quantisation_rms,execution_rms,rms_command,max_step_jump,closure_error
run,,,,,,,,
cardinal,cardinal_4,60.865,121.848,61.067,2.754,26.484,72.0,2.853
cardinal_diagonal,cardinal_8,32.612,74.045,32.644,2.956,26.397,45.0,2.853
free_form,none,3.517,10.426,0.000,3.517,26.365,2.0,0.000
ik,none,3.517,10.426,0.000,3.517,26.365,2.0,0.000


## Shape metrics, and why they mislead

Direction quantisation is radius-preserving, so every reconstructed point sits
on the commanded radius. Aspect ratio and circle-fit RMS therefore score all
four strategies as near-perfect circles while point-to-point error differs by
more than an order of magnitude. That is why `total_rms` is the primary metric
here.

In [3]:
names = [s.value for s in config.strategies]
metrics.loc[names, ["radial_rms", "circle_fit_rms", "aspect_ratio",
                    "closure_error", "decode_invalid_steps"]].round(3)

,radial_rms,circle_fit_rms,aspect_ratio,closure_error,decode_invalid_steps
run,,,,,
cardinal,1.912,0.323,1.004,2.853,0
cardinal_diagonal,2.167,0.768,1.004,2.853,0
free_form,3.143,1.693,0.997,0.000,0
ik,3.143,1.693,0.997,0.000,0


## Figures

In [4]:
for plot in (plot_reconstructed_paths(samples, config),
             plot_strategy_comparison(metrics, config),
             plot_motor_commands(samples, config)):
    print("saved", plot.name)

saved reconstructed_paths_by_strategy.png
saved strategy_comparison.png
saved motor_commands_by_strategy.png
